# Script de Inyeccion de datos
En este archivo, inyectamos los datos homologados y normalizados

### IMPORTACIONES DE BIBLIOTECAS NECESARIAS

In [2]:
import polars as pl
import sqlalchemy as sa
from IPython.display import display
from pydantic import BaseModel

### Constantes del Notebook
Aqui se definen las constantes que se necesiten para subir los archivos

In [39]:
FILE_PATH = './layouts/oferta_academicav2.2.xlsx'  # cambiar por ruta del archivo que se desea leer
SHEETS_NAMES = ['area', 'escuela', 'carrera', 'habilidad', "carrera_habilidad", "criterio"]  # nombre de las hojas que se desean procesar
DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = "password"
DB_CONECTION = "athena"
DIR_SAVE = "load_data.sql"

### Modelos de pydantic

en este apartado estan los modelos para subir a la base de datos los datos correspondientes sin romper la tabla

In [12]:
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Escuela(BaseModel):
    id_escuela: int
    nombre_escuela: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str
    id_area: int

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class criterios(BaseModel):
    id_criterio: int
    descripcion: str
    id_habilidad: int

### Funciones de procesamiento

aqui estan todas las funciones para procesar el excel, dependiendo de la hoja lo procesa de manera distinta

In [37]:
class SQLGenerator:
    def carrera(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO carrera (id_carrera, nombre_carrera, id_area) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_carrera']}, '{row['nombre_carrera']}', {row['id_area']})")
        sql += ",\n".join(values) + ";"
        return sql

    def area(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO area (id_area, nombre_area) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_area']}, '{row['nombre_area']}')")
        sql += ",\n".join(values) + ";"
        return sql

    def escuela(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO escuela (id_escuela, nombre_escuela) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_escuela']}, '{row['nombre_escuela']}')")
        sql += ",\n".join(values) + ";"
        return sql

    def habilidad(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO habilidad (id_habilidad, descripcion) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_habilidad']}, '{row['descripcion']}')")
        sql += ",\n".join(values) + ";"
        return sql

    def carrera_habilidad(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO carrera_habilidad (id_carrera, id_habilidad) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_carrera']}, {row['id_habilidad']})")
        sql += ",\n".join(values) + ";"
        return sql

    def criterio(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO criterio (id_criterio, descripcion, id_habilidad) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_criterio']}, '{row['nombre_criterio']}', {row['id_habilidad']})")
        sql += ",\n".join(values) + ";"
        return sql

def db_conection():
    try:
        engine = sa.create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_CONECTION}')
        print("Database connection successful.")
        return engine
    except Exception as e:
        print(f"Error connecting to the database: {e}")
        return None

def read_excel_file(file_path: str, sheet_name: str) -> pl.DataFrame:
    df_sheet = pl.read_excel(file_path, sheet_name=sheet_name)
    return df_sheet

def view_excel_sheets(df: pl.DataFrame, sheet_name: str):
    print(f"Sheet: {sheet_name}")
    display(df[:5])

def ask_submit_data(df: pl.DataFrame, sheet_name: str):
    SQLGen = SQLGenerator()
    sql_archive = getattr(SQLGen, sheet_name)(df)
    print(f"SQL for {sheet_name}:\n{sql_archive}")
    return sql_archive

def save_text_to_file(text: str, filename: str):
    with open(filename, 'w') as f:
        f.write(text)


### Ejecucion

aqui se ejecuta el script, haciendo que se procese y se suba la informacion

In [42]:
if __name__ == "__main__":
    sql_data = []
    for sheet_name in SHEETS_NAMES:
        df = read_excel_file(FILE_PATH, sheet_name)
        view_excel_sheets(df, sheet_name)
        sql_text = ask_submit_data(df, sheet_name)
        sql_data.append(sql_text)
    save_text_to_file("\n\n".join(sql_data), DIR_SAVE)


Sheet: area


/var/folders/5c/1h_2rjpj3wg1lps5yq7kg2740000gn/T/ipykernel_56357/3370824070.py:60: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df_sheet = pl.read_excel(file_path, sheet_name=sheet_name)


id_area,nombre_area
i64,str
1,"""Ingeniería y Ciencias Físico M…"
2,"""Ciencias Médico Biológicas"""
3,"""Ciencias Sociales y Administra…"


SQL for area:
INSERT INTO area (id_area, nombre_area) VALUES (1, 'Ingeniería y Ciencias Físico Matemáticas'),
(2, 'Ciencias Médico Biológicas'),
(3, 'Ciencias Sociales y Administrativas');
Sheet: escuela


id_escuela,nombre_escuela
i64,str
1,"""CICS Unidad Milpa Alta"""
2,"""CICS Unidad Santo Tomás"""
3,"""ENBA"""
4,"""ENCB"""
5,"""ENMyH"""


...

SQL for habilidad:
INSERT INTO habilidad (id_habilidad, descripcion) VALUES (1, 'Diseño, construcción, mantenimiento y operación de aeronaves y sistemas aeronáuticos'),
(2, 'Proyección y puesta en operación de plantas e infraestructura de soporte aéreo'),
(3, 'Investigación, adaptación y desarrollo de nuevas tecnologías aeronáuticas'),
(4, 'Planeación, dirección y gestión de empresas de mantenimiento y servicios aeronáuticos'),
(5, 'Aplicación de normatividad técnica, jurídica, ética y de seguridad aeronáutica'),
(6, 'Comunicación técnica oral y escrita en español e inglés'),
(7, 'Pensamiento analítico, lógico, crítico y creativo para la solución de problemas de ingeniería'),
(8, 'Extracción de conocimiento implícito, patrones y anomalías en grandes volúmenes de datos (Big Data)'),
(9, 'Aplicación de inteligencia artificial, aprendizaje de máquina (Machine Learning) y estadística avanzada'),
(10, 'Diseño y gestión de sistemas de bases de datos para toma de decisiones directivas'),
(11,

id_carrera,id_habilidad
i64,i64
1,1
1,2
1,3
1,4
1,5


SQL for carrera_habilidad:
INSERT INTO carrera_habilidad (id_carrera, id_habilidad) VALUES (1, 1),
(1, 2),
(1, 3),
(1, 4),
(1, 5),
(1, 6),
(1, 7),
(2, 7),
(2, 8),
(2, 9),
(2, 10),
(2, 11),
(2, 12),
(3, 6),
(3, 13),
(3, 14),
(3, 15),
(3, 16),
(3, 17),
(4, 18),
(4, 19),
(4, 20),
(4, 21),
(4, 22),
(4, 23),
(5, 12),
(5, 22),
(5, 24),
(5, 25),
(5, 26),
(5, 27),
(6, 28),
(6, 29),
(6, 30),
(6, 31),
(6, 32),
(7, 7),
(7, 21),
(7, 33),
(7, 34),
(7, 35),
(8, 23),
(8, 35),
(8, 36),
(8, 37),
(8, 38),
(9, 17),
(9, 21),
(9, 39),
(9, 40),
(9, 41),
(10, 22),
(10, 23),
(10, 42),
(10, 43),
(10, 44),
(11, 7),
(11, 45),
(11, 46),
(11, 47),
(11, 48),
(12, 26),
(12, 49),
(12, 50),
(12, 51),
(12, 52),
(13, 20),
(13, 21),
(13, 53),
(13, 54),
(13, 55),
(14, 23),
(14, 56),
(14, 57),
(14, 58),
(14, 59),
(15, 9),
(15, 12),
(15, 60),
(15, 61),
(15, 62),
(16, 23),
(16, 63),
(16, 64),
(16, 65),
(16, 66),
(17, 6),
(17, 26),
(17, 67),
(17, 68),
(17, 69),
(18, 7),
(18, 70),
(18, 71),
(18, 72),
(18, 73),
(19, 45),
(19, 7

id_criterio,nombre_criterio,id_habilidad
i64,str,i64
1,"""Diseño""",1
2,"""Construcción""",1
3,"""Mantenimiento""",1
4,"""Proyección""",2
5,"""Puesta""",2


SQL for criterio:
INSERT INTO criterio (id_criterio, descripcion, id_habilidad) VALUES (1, 'Diseño', 1),
(2, 'Construcción', 1),
(3, 'Mantenimiento', 1),
(4, 'Proyección', 2),
(5, 'Puesta', 2),
(6, 'Operación de plantas', 2),
(7, 'Investigación', 3),
(8, 'Adaptación', 3),
(9, 'Desarrollo', 3),
(10, 'Planeación', 4),
(11, 'Dirección', 4),
(12, 'Gestión de empresas', 4),
(13, 'Aplicación de normatividad', 5),
(14, 'Jurídica', 5),
(15, 'Seguridad aeronáutica', 5),
(16, 'Comunicación técnica oral', 6),
(17, 'Escrita', 6),
(18, 'Español', 6),
(19, 'Pensamiento analítico', 7),
(20, 'Lógico', 7),
(21, 'Crítico', 7),
(22, 'Extracción de conocimiento', 8),
(23, 'Patrones', 8),
(24, 'Anomalías', 8),
(25, 'Aplicación de inteligencia', 9),
(26, 'Aprendizaje de máquina', 9),
(27, 'Estadística avanzada', 9),
(28, 'Gestión de sistemas', 10),
(29, 'Toma de decisiones', 10),
(30, 'Bases de datos', 10),
(31, 'Modelado matemático predictivo', 11),
(32, 'Finanzas', 11),
(33, 'Ciencia', 11),
(34, 'Manejo d